# Libraries

Before run this notebook you need install the dependencies. See the `requirements.txt` file.


In [1]:
from functions_victor_project import *

import multiprocessing
import numpy as np
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import matplotlib.pyplot as plt
import pickle
import dill
from scipy.integrate import odeint
from UQpy.distributions import Uniform, Normal, JointIndependent #, Lognormal
from UQpy.distributions.collection.Lognormal import Lognormal
from UQpy.surrogates import *
from sklearn.metrics import mean_squared_error, r2_score

# Statistics of the input variables

- $R$: Resistance variable  
- $S$: Demand variable  
- $Z_1$ and $Z_2$: Latent variables  

State Limit Function:

$$
g(R, S, t) = k_{factor}(t) \cdot \frac{R}{Z_1} - S \cdot Z_2
$$

**Table 1:** Moments, distributions, and parameters of the considered variables for the $R$ and $S$ problem.

| Variable | Distribution | Mean | Std. Deviation |  
|----------|--------------|------|----------------|  
| $R$    | Normal     | 5.0  | 0.8            |  
| $S$    | Normal     | 2.0  | 0.6            |  
| $Z_1$ | Lognormal   | 1.0  | 0.028          |  
| $Z_2$ | Lognormal   | 1.0  | 0.096          |  

To simulate $Z_1$ and $Z_2$ we are used `scipy` library. You can see the file `glam_example_1.py` and how to generate state limit results. 

### Transformation to scipy parameters

**Shape parameter (s):**
$$
s = \sqrt{\ln\left(1 + \left(\frac{\sigma}{\mu}\right)^2\right)}
$$

**Scale parameter:**
$$
\text{scale} = \frac{\mu}{\sqrt{1 + \left(\frac{\sigma}{\mu}\right)^2}}
$$

Where:
- $\mu$ = mean of the lognormal distribution  
- $\sigma$ = standard deviation of the lognormal distribution
- $s$ = shape parameter for `scipy.stats.lognorm`
- $\text{scale}$ = scale parameter for `scipy.stats.lognorm`

# Full process for time PCE

In [2]:
n_samples = 2000
n_latent_samples = 10000
time = [0, 10, 20, 30]

# Prepare input arguments in tuple format [(0, 1000, 5000), (10, 1000, 5000), ...]
inputs = [(k, n_samples, n_latent_samples) for k in time]

# Execution
with multiprocessing.Pool() as pool:
    # starmap desempacota a tupla de inputs para a função
    resultados = pool.starmap(execute_parallel_process, inputs)
    
# Results
df = pd.DataFrame(resultados)
df.to_excel('validation_results_toy_problem.xlsx', index=False)
df

,Time,R2 (Lambda 1),R2 (Lambda 2),R2 (Lambda 3),R2 (Lambda 4)
0,0,0.999992,0.992418,0.606323,0.588812
1,10,0.999992,0.992060,0.583365,0.591284
2,20,0.999992,0.993262,0.555159,0.558628
3,30,0.999991,0.993347,0.579651,0.539030


# Latex export

In [3]:
codigo_latex = df.to_latex(index=False, float_format="%.4f", caption="Statistical validation results of the PCE metamodel at different time steps.", label="tab:validation_results")
codigo_latex

'\\begin{table}\n\\caption{Statistical validation results of the PCE metamodel at different time steps.}\n\\label{tab:validation_results}\n\\begin{tabular}{rrrrr}\n\\toprule\nTime & R2 (Lambda 1) & R2 (Lambda 2) & R2 (Lambda 3) & R2 (Lambda 4) \\\\\n\\midrule\n0 & 1.0000 & 0.9924 & 0.6063 & 0.5888 \\\\\n10 & 1.0000 & 0.9921 & 0.5834 & 0.5913 \\\\\n20 & 1.0000 & 0.9933 & 0.5552 & 0.5586 \\\\\n30 & 1.0000 & 0.9933 & 0.5797 & 0.5390 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{table}\n'